## Setup: install required packages for this notebook

If imports fail (e.g., ModuleNotFoundError for scikit-learn), run the next cell to install dependencies into the active kernel. After installation, you may need to restart the kernel.

In [17]:
# Install dependencies (run if you see ModuleNotFoundError)
import sys, subprocess
pkgs = [
    'scikit-learn',
    'xgboost',
    'ta',
    'plotly'
]
for p in pkgs:
    try:
        __import__(p.replace('-', '_'))
    except Exception:
        print(f'Installing {p}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', p])
print('Done. If imports still fail, restart the kernel and rerun.')

Installing scikit-learn...
Done. If imports still fail, restart the kernel and rerun.


# Model Comparison Presentation
This notebook compares multiple models (XGBoost, ANN/MLP, SVM, Naive Bayes) on both regression and classification tasks using your existing src pipeline.
- Uses TimeSeriesSplit (5 folds) and reports per-fold metrics + summary.
- Loads a single ticker and horizon to keep the demo fast.
- Reuses feature engineering from `src.features` and labeling from `src.labeling`.

Tip: Run the cells in order. Adjust the ticker, date range, and horizon in the next cell if needed.

In [18]:
# Imports & settings
import os, sys
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_score, recall_score, f1_score)
from xgboost import XGBRegressor, XGBClassifier
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.svm import SVR, SVC
from sklearn.naive_bayes import GaussianNB

# make sure we can import src/* when running this notebook
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    # if running from reports/ ensure project root is in path
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.features import add_basic_features
from src.labeling import create_reg_target, create_class_labels

np.random.seed(42)

# Config (edit here for your demo)
TICKERS = ['BAC', 'NKE', 'TSLA']
START = '2018-01-01'
END = None  # to present
HORIZON = 1  # 1 day ahead
N_SPLITS = 5

# Ensure date boundaries are UTC-aware Timestamps to avoid tz-naive/aware comparison errors
START_DT = pd.to_datetime(START, errors='coerce', utc=True)
END_DT = pd.Timestamp.now(tz='UTC') if END is None else pd.to_datetime(END, errors='coerce', utc=True)


def prepare_numeric(df):
    # coerce key columns to numeric if present
    for c in ['Open','High','Low','Close','Volume']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

In [19]:
# Load, engineer, and evaluate REGRESSION for each ticker
reg_results_all = {}
for TICKER in TICKERS:
    DATA_CSV = ROOT / 'data' / f'{TICKER}.csv'
    assert DATA_CSV.exists(), f'Missing data file: {DATA_CSV}'

    # Load & prepare
    df = pd.read_csv(DATA_CSV)
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce', utc=True)
    df = df[(df['Date'] >= START_DT) & (df['Date'] <= END_DT)].reset_index(drop=True)
    df = prepare_numeric(df)

    # Features & target
    df_reg = create_reg_target(df.copy(), horizon=HORIZON)
    df_reg = add_basic_features(df_reg)
    non_features_reg = {'Date','next_close'}
    features_reg = [c for c in df_reg.columns if c not in non_features_reg and np.issubdtype(df_reg[c].dtype, np.number)]
    df_reg = df_reg.dropna(subset=features_reg + ['next_close']).reset_index(drop=True)
    Xr = df_reg[features_reg]
    yr = df_reg['next_close']

    display(pd.DataFrame({'ticker':[TICKER], 'rows':[len(df_reg)], 'n_features':[len(features_reg)]}))

    # Evaluate regressors
    tscv = TimeSeriesSplit(n_splits=N_SPLITS)
    reg_models = {
        'XGBRegressor': Pipeline([('scaler', StandardScaler()), ('xgb', XGBRegressor(n_estimators=150, max_depth=5, learning_rate=0.1, subsample=0.9, colsample_bytree=1.0, random_state=42))]),
        'MLPRegressor': Pipeline([('scaler', StandardScaler()), ('mlp', MLPRegressor(hidden_layer_sizes=(64,32), alpha=1e-3, learning_rate_init=1e-3, max_iter=500, early_stopping=True, random_state=42))]),
        'SVR': Pipeline([('scaler', StandardScaler()), ('svr', SVR())])
    }

    reg_results = {}
    for name, pipe in reg_models.items():
        fold_rows = []
        for i, (tr, te) in enumerate(tscv.split(Xr), start=1):
            Xtr, Xte = Xr.iloc[tr], Xr.iloc[te]
            ytr, yte = yr.iloc[tr], yr.iloc[te]
            # Fit and predict
            pipe.fit(Xtr, ytr)
            pred = pipe.predict(Xte)
            # Naive baseline: predict current Close for next_close
            baseline_pred = df_reg['Close'].iloc[te].to_numpy()
            # Fold test window
            test_start = pd.to_datetime(df_reg['Date'].iloc[te].min())
            test_end = pd.to_datetime(df_reg['Date'].iloc[te].max())
            # Metrics
            mae = float(mean_absolute_error(yte, pred))
            rmse = float(np.sqrt(mean_squared_error(yte, pred)))
            r2 = float(r2_score(yte, pred))
            mae_base = float(mean_absolute_error(yte, baseline_pred))
            rmse_base = float(np.sqrt(mean_squared_error(yte, baseline_pred)))
            fold_rows.append({
                'ticker': TICKER,
                'model': name,
                'fold': i,
                'test_start': str(test_start.date()) if not pd.isna(test_start) else None,
                'test_end': str(test_end.date()) if not pd.isna(test_end) else None,
                'mae': mae,
                'rmse': rmse,
                'r2': r2,
                'mae_base': mae_base,
                'rmse_base': rmse_base
            })
        res_df = pd.DataFrame(fold_rows)
        reg_results[name] = res_df
        summary_mean = res_df.agg({'mae':'mean','rmse':'mean','r2':'mean','mae_base':'mean','rmse_base':'mean'}).to_frame().T.assign(ticker=TICKER, model=name, fold='mean')
        summary_median = res_df.agg({'mae':'median','rmse':'median','r2':'median','mae_base':'median','rmse_base':'median'}).to_frame().T.assign(ticker=TICKER, model=name, fold='median')
        display(pd.concat([res_df, summary_mean, summary_median], ignore_index=True))
    reg_results_all[TICKER] = reg_results

,ticker,rows,n_features
0,BAC,1887,36


,ticker,model,fold,test_start,test_end,mae,rmse,r2,mae_base,rmse_base
0,BAC,XGBRegressor,1,2019-05-24,2020-08-20,1.365163,1.854557,0.812489,0.519490,0.766708
1,BAC,XGBRegressor,2,2020-08-21,2021-11-17,4.032436,5.487446,0.420947,0.475032,0.614704
2,BAC,XGBRegressor,3,2021-11-18,2023-02-17,0.781059,1.031225,0.961254,0.551274,0.731357
3,BAC,XGBRegressor,4,2023-02-21,2024-05-20,0.443705,0.578133,0.973644,0.370032,0.496849
4,BAC,XGBRegressor,5,2024-05-21,2025-08-21,0.590101,0.818945,0.939811,0.486720,0.720234
5,BAC,XGBRegressor,mean,NaN,NaN,1.442493,1.954061,0.821629,0.480510,0.665970
6,BAC,XGBRegressor,median,NaN,NaN,0.781059,1.031225,0.939811,0.486720,0.720234


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


,ticker,model,fold,test_start,test_end,mae,rmse,r2,mae_base,rmse_base
0,BAC,MLPRegressor,1,2019-05-24,2020-08-20,12.816700,17.673329,-16.028802,0.519490,0.766708
1,BAC,MLPRegressor,2,2020-08-21,2021-11-17,4.235475,5.728367,0.368986,0.475032,0.614704
2,BAC,MLPRegressor,3,2021-11-18,2023-02-17,2.215585,2.818170,0.710626,0.551274,0.731357
3,BAC,MLPRegressor,4,2023-02-21,2024-05-20,1.444102,1.852423,0.729417,0.370032,0.496849
4,BAC,MLPRegressor,5,2024-05-21,2025-08-21,1.192679,1.672032,0.749103,0.486720,0.720234
5,BAC,MLPRegressor,mean,NaN,NaN,4.380908,5.948864,-2.694134,0.480510,0.665970
6,BAC,MLPRegressor,median,NaN,NaN,2.215585,2.818170,0.710626,0.486720,0.720234


,ticker,model,fold,test_start,test_end,mae,rmse,r2,mae_base,rmse_base
0,BAC,SVR,1,2019-05-24,2020-08-20,2.708818,3.536388,0.318184,0.519490,0.766708
1,BAC,SVR,2,2020-08-21,2021-11-17,7.441267,10.068331,-0.949362,0.475032,0.614704
2,BAC,SVR,3,2021-11-18,2023-02-17,2.422407,3.640999,0.516979,0.551274,0.731357
3,BAC,SVR,4,2023-02-21,2024-05-20,0.514133,0.652099,0.966469,0.370032,0.496849
4,BAC,SVR,5,2024-05-21,2025-08-21,1.691793,2.571993,0.406329,0.486720,0.720234
5,BAC,SVR,mean,NaN,NaN,2.955684,4.093962,0.251720,0.480510,0.665970
6,BAC,SVR,median,NaN,NaN,2.422407,3.536388,0.406329,0.486720,0.720234


...

,ticker,model,fold,test_start,test_end,mae,rmse,r2,mae_base,rmse_base
0,NKE,SVR,1,2019-05-24,2020-08-20,11.433128,14.272992,-2.017508,1.377293,1.984473
1,NKE,SVR,2,2020-08-21,2021-11-17,57.239266,60.100750,-12.926964,1.620764,2.405562
2,NKE,SVR,3,2021-11-18,2023-02-17,8.515307,12.096038,0.688770,2.150541,2.854054
3,NKE,SVR,4,2023-02-21,2024-05-20,2.786522,3.786225,0.859721,1.267325,1.823898
4,NKE,SVR,5,2024-05-21,2025-08-21,19.707612,25.117465,-5.767423,1.137771,1.926325
5,NKE,SVR,mean,NaN,NaN,19.936367,23.074694,-3.832681,1.510739,2.198862
6,NKE,SVR,median,NaN,NaN,11.433128,14.272992,-2.017508,1.377293,1.984473


,ticker,rows,n_features
0,TSLA,1887,36


,ticker,model,fold,test_start,test_end,mae,rmse,r2,mae_base,rmse_base
0,TSLA,XGBRegressor,1,2019-05-24,2020-08-20,19.546900,32.469354,-0.321210,1.424558,2.542127
1,TSLA,XGBRegressor,2,2020-08-21,2021-11-17,97.855075,112.967087,-2.855561,5.976918,8.565387
2,TSLA,XGBRegressor,3,2021-11-18,2023-02-17,12.280327,15.829704,0.946057,8.279693,10.994704
3,TSLA,XGBRegressor,4,2023-02-21,2024-05-20,5.759069,7.667109,0.956436,5.007994,6.783017
4,TSLA,XGBRegressor,5,2024-05-21,2025-08-21,15.051636,23.341207,0.890433,9.059204,12.444431
5,TSLA,XGBRegressor,mean,NaN,NaN,30.098602,38.454892,-0.076769,5.949673,8.265933
6,TSLA,XGBRegressor,median,NaN,NaN,15.051636,23.341207,0.890433,5.976918,8.565387


,ticker,model,fold,test_start,test_end,mae,rmse,r2,mae_base,rmse_base
0,TSLA,MLPRegressor,1,2019-05-24,2020-08-20,76.172740,122.008082,-17.655277,1.424558,2.542127
1,TSLA,MLPRegressor,2,2020-08-21,2021-11-17,39.732818,43.958552,0.416191,5.976918,8.565387
2,TSLA,MLPRegressor,3,2021-11-18,2023-02-17,19.433309,24.017201,0.875826,8.279693,10.994704
3,TSLA,MLPRegressor,4,2023-02-21,2024-05-20,14.161391,17.189697,0.781020,5.007994,6.783017
4,TSLA,MLPRegressor,5,2024-05-21,2025-08-21,12.688055,16.983454,0.941992,9.059204,12.444431
5,TSLA,MLPRegressor,mean,NaN,NaN,32.437663,44.831397,-2.928050,5.949673,8.265933
6,TSLA,MLPRegressor,median,NaN,NaN,19.433309,24.017201,0.781020,5.976918,8.565387


,ticker,model,fold,test_start,test_end,mae,rmse,r2,mae_base,rmse_base
0,TSLA,SVR,1,2019-05-24,2020-08-20,21.777792,34.204646,-0.466205,1.424558,2.542127
1,TSLA,SVR,2,2020-08-21,2021-11-17,184.859085,193.728796,-10.338929,5.976918,8.565387
2,TSLA,SVR,3,2021-11-18,2023-02-17,101.902993,119.599141,-2.079236,8.279693,10.994704
3,TSLA,SVR,4,2023-02-21,2024-05-20,11.022205,15.789664,0.815237,5.007994,6.783017
4,TSLA,SVR,5,2024-05-21,2025-08-21,70.862794,96.800978,-0.884487,9.059204,12.444431
5,TSLA,SVR,mean,NaN,NaN,78.084974,92.024645,-2.590724,5.949673,8.265933
6,TSLA,SVR,median,NaN,NaN,70.862794,96.800978,-0.884487,5.976918,8.565387


In [16]:
# Evaluate REGRESSORS (XGB, ANN/MLP, SVM) with 5-fold TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=N_SPLITS)
reg_models = {
    'XGBRegressor': Pipeline([('scaler', StandardScaler()), ('xgb', XGBRegressor(n_estimators=150, max_depth=5, learning_rate=0.1, subsample=0.9, colsample_bytree=1.0, random_state=42))]),
    'MLPRegressor': Pipeline([('scaler', StandardScaler()), ('mlp', MLPRegressor(hidden_layer_sizes=(64,32), alpha=1e-3, learning_rate_init=1e-3, max_iter=500, early_stopping=True, random_state=42))]),
    'SVR': Pipeline([('scaler', StandardScaler()), ('svr', SVR())])
}

reg_results = {}
for name, pipe in reg_models.items():
    fold_rows = []
    for i, (tr, te) in enumerate(tscv.split(Xr), start=1):
        Xtr, Xte = Xr.iloc[tr], Xr.iloc[te]
        ytr, yte = yr.iloc[tr], yr.iloc[te]
        # Fit and predict
        pipe.fit(Xtr, ytr)
        pred = pipe.predict(Xte)
        # Naive baseline: predict current Close for next_close
        baseline_pred = df_reg['Close'].iloc[te].to_numpy()
        # Fold test window
        test_start = pd.to_datetime(df_reg['Date'].iloc[te].min())
        test_end = pd.to_datetime(df_reg['Date'].iloc[te].max())
        # Metrics
        mae = float(mean_absolute_error(yte, pred))
        rmse = float(np.sqrt(mean_squared_error(yte, pred)))
        r2 = float(r2_score(yte, pred))
        mae_base = float(mean_absolute_error(yte, baseline_pred))
        rmse_base = float(np.sqrt(mean_squared_error(yte, baseline_pred)))
        fold_rows.append({
            'fold': i,
            'test_start': str(test_start.date()) if not pd.isna(test_start) else None,
            'test_end': str(test_end.date()) if not pd.isna(test_end) else None,
            'mae': mae,
            'rmse': rmse,
            'r2': r2,
            'mae_base': mae_base,
            'rmse_base': rmse_base
        })
    res_df = pd.DataFrame(fold_rows)
    reg_results[name] = res_df
    summary_mean = res_df.agg({'mae':'mean','rmse':'mean','r2':'mean','mae_base':'mean','rmse_base':'mean'}).to_frame().T.assign(fold='mean')
    summary_median = res_df.agg({'mae':'median','rmse':'median','r2':'median','mae_base':'median','rmse_base':'median'}).to_frame().T.assign(fold='median')
    display(pd.concat([res_df, summary_mean, summary_median], ignore_index=True))

,fold,test_start,test_end,mae,rmse,r2,mae_base,rmse_base
0,1,2018-03-09,2018-03-23,0.771350,0.963226,0.377200,0.608409,0.806537
1,2,2018-03-26,2018-04-10,0.607158,0.795004,-0.795560,0.566137,0.648050
2,3,2018-04-11,2018-04-25,1.280315,1.531305,-0.107786,0.508636,0.724255
3,4,2018-04-26,2018-05-10,1.520228,1.724924,0.459954,0.706136,0.906505
4,5,2018-05-11,2018-05-25,1.508120,1.653260,-77.347928,0.228864,0.262402
5,mean,NaN,NaN,1.137434,1.333544,-15.482824,0.523636,0.669550
6,median,NaN,NaN,1.280315,1.531305,-0.107786,0.566137,0.724255


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


,fold,test_start,test_end,mae,rmse,r2,mae_base,rmse_base
0,1,2018-03-09,2018-03-23,40.257551,43.278145,-1256.273137,0.608409,0.806537
1,2,2018-03-26,2018-04-10,17.458005,18.570618,-978.747964,0.566137,0.648050
2,3,2018-04-11,2018-04-25,6.600571,7.588261,-26.203006,0.508636,0.724255
3,4,2018-04-26,2018-05-10,13.604633,15.353645,-41.787211,0.706136,0.906505
4,5,2018-05-11,2018-05-25,45.052850,48.400508,-67148.833297,0.228864,0.262402
5,mean,NaN,NaN,24.594722,26.638235,-13890.368923,0.523636,0.669550
6,median,NaN,NaN,17.458005,18.570618,-978.747964,0.566137,0.724255


,fold,test_start,test_end,mae,rmse,r2,mae_base,rmse_base
0,1,2018-03-09,2018-03-23,0.989392,1.236209,-0.025832,0.608409,0.806537
1,2,2018-03-26,2018-04-10,1.110178,1.249877,-3.438091,0.566137,0.648050
2,3,2018-04-11,2018-04-25,1.181320,1.411144,0.059248,0.508636,0.724255
3,4,2018-04-26,2018-05-10,2.464187,2.857261,-0.481804,0.706136,0.906505
4,5,2018-05-11,2018-05-25,2.667869,2.722663,-211.487286,0.228864,0.262402
5,mean,NaN,NaN,1.682589,1.895431,-43.074753,0.523636,0.669550
6,median,NaN,NaN,1.181320,1.411144,-0.481804,0.566137,0.724255


In [12]:
# Load, engineer, and evaluate CLASSIFICATION for each ticker
cls_results_all = {}
for TICKER in TICKERS:
    DATA_CSV = ROOT / 'data' / f'{TICKER}.csv'
    assert DATA_CSV.exists(), f'Missing data file: {DATA_CSV}'

    # Use already-loaded df from regression loop if ticker same? For clarity, reload.
    df = pd.read_csv(DATA_CSV)
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce', utc=True)
    df = df[(df['Date'] >= START_DT) & (df['Date'] <= END_DT)].reset_index(drop=True)
    df = prepare_numeric(df)

    dfc = create_class_labels(df.copy(), horizon=HORIZON)
    dfc = add_basic_features(dfc)
    non_features_cls = {'Date','next_close','ret_next','label'}
    features_cls = [c for c in dfc.columns if c not in non_features_cls and np.issubdtype(dfc[c].dtype, np.number)]
    dfc = dfc.dropna(subset=features_cls + ['label']).reset_index(drop=True)
    Xc = dfc[features_cls]
    yc = (dfc['label'] + 1).astype(int)  # -1,0,1 -> 0,1,2

    display(pd.DataFrame({'ticker':[TICKER], 'rows':[len(dfc)], 'n_features':[len(features_cls)]}))

    tscv = TimeSeriesSplit(n_splits=N_SPLITS)
    cls_models = {
        'XGBClassifier': Pipeline([('scaler', StandardScaler()), ('xgb', XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.1, subsample=0.9, colsample_bytree=1.0, use_label_encoder=False, eval_metric='mlogloss', random_state=42))]),
        'MLPClassifier': Pipeline([('scaler', StandardScaler()), ('mlp', MLPClassifier(hidden_layer_sizes=(64,32), alpha=1e-3, learning_rate_init=1e-3, max_iter=500, early_stopping=True, random_state=42))]),
        'SVC': Pipeline([('scaler', StandardScaler()), ('svc', SVC(probability=True, random_state=42))]),
        'GaussianNB': Pipeline([('scaler', StandardScaler()), ('nb', GaussianNB())])
    }

    cls_results = {}
    for name, pipe in cls_models.items():
        fold_rows = []
        for i, (tr, te) in enumerate(tscv.split(Xc), start=1):
            Xtr, Xte = Xc.iloc[tr], Xc.iloc[te]
            ytr, yte = yc.iloc[tr], yc.iloc[te]
            pipe.fit(Xtr, ytr)
            pred = pipe.predict(Xte)
            # back to -1/0/1
            pred_orig = pred - 1
            yte_orig = yte - 1
            # Fold test window
            test_start = pd.to_datetime(dfc['Date'].iloc[te].min())
            test_end = pd.to_datetime(dfc['Date'].iloc[te].max())
            fold_rows.append({
                'ticker': TICKER,
                'model': name,
                'fold': i,
                'test_start': str(test_start.date()) if not pd.isna(test_start) else None,
                'test_end': str(test_end.date()) if not pd.isna(test_end) else None,
                'accuracy': float(accuracy_score(yte_orig, pred_orig)),
                'precision_macro': float(precision_score(yte_orig, pred_orig, average='macro', zero_division=0)),
                'recall_macro': float(recall_score(yte_orig, pred_orig, average='macro', zero_division=0)),
                'f1_macro': float(f1_score(yte_orig, pred_orig, average='macro', zero_division=0))
            })
        res_df = pd.DataFrame(fold_rows)
        cls_results[name] = res_df
        summary_mean = res_df.agg({'accuracy':'mean','precision_macro':'mean','recall_macro':'mean','f1_macro':'mean'}).to_frame().T.assign(ticker=TICKER, model=name, fold='mean')
        summary_median = res_df.agg({'accuracy':'median','precision_macro':'median','recall_macro':'median','f1_macro':'median'}).to_frame().T.assign(ticker=TICKER, model=name, fold='median')
        display(pd.concat([res_df, summary_mean, summary_median], ignore_index=True))
    cls_results_all[TICKER] = cls_results

(68, 36)

In [15]:
# Evaluate CLASSIFIERS (XGB, ANN/MLP, SVM, Naive Bayes) with 5-fold TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=N_SPLITS)
cls_models = {
    'XGBClassifier': Pipeline([('scaler', StandardScaler()), ('xgb', XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.1, subsample=0.9, colsample_bytree=1.0, use_label_encoder=False, eval_metric='mlogloss', random_state=42))]),
    'MLPClassifier': Pipeline([('scaler', StandardScaler()), ('mlp', MLPClassifier(hidden_layer_sizes=(64,32), alpha=1e-3, learning_rate_init=1e-3, max_iter=500, early_stopping=True, random_state=42))]),
    'SVC': Pipeline([('scaler', StandardScaler()), ('svc', SVC(probability=True, random_state=42))]),
    'GaussianNB': Pipeline([('scaler', StandardScaler()), ('nb', GaussianNB())])
}

cls_results = {}
for name, pipe in cls_models.items():
    fold_rows = []
    for i, (tr, te) in enumerate(tscv.split(Xc), start=1):
        Xtr, Xte = Xc.iloc[tr], Xc.iloc[te]
        ytr, yte = yc.iloc[tr], yc.iloc[te]
        pipe.fit(Xtr, ytr)
        pred = pipe.predict(Xte)
        # back to -1/0/1
        pred_orig = pred - 1
        yte_orig = yte - 1
        # Fold test window
        test_start = pd.to_datetime(dfc['Date'].iloc[te].min())
        test_end = pd.to_datetime(dfc['Date'].iloc[te].max())
        fold_rows.append({
            'fold': i,
            'test_start': str(test_start.date()) if not pd.isna(test_start) else None,
            'test_end': str(test_end.date()) if not pd.isna(test_end) else None,
            'accuracy': float(accuracy_score(yte_orig, pred_orig)),
            'precision_macro': float(precision_score(yte_orig, pred_orig, average='macro', zero_division=0)),
            'recall_macro': float(recall_score(yte_orig, pred_orig, average='macro', zero_division=0)),
            'f1_macro': float(f1_score(yte_orig, pred_orig, average='macro', zero_division=0))
        })
    res_df = pd.DataFrame(fold_rows)
    cls_results[name] = res_df
    summary_mean = res_df.agg({'accuracy':'mean','precision_macro':'mean','recall_macro':'mean','f1_macro':'mean'}).to_frame().T.assign(fold='mean')
    summary_median = res_df.agg({'accuracy':'median','precision_macro':'median','recall_macro':'median','f1_macro':'median'}).to_frame().T.assign(fold='median')
    display(pd.concat([res_df, summary_mean, summary_median], ignore_index=True))

C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\training.py:183: UserWarning: [22:51:44] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\training.py:183: UserWarning: [22:51:44] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\training.py:183: UserWarning: [22:51:45] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are

,fold,test_start,test_end,accuracy,precision_macro,recall_macro,f1_macro
0,1,2018-03-09,2018-03-23,0.272727,0.407407,0.388889,0.216450
1,2,2018-03-26,2018-04-10,0.181818,0.222222,0.416667,0.288889
2,3,2018-04-11,2018-04-25,0.181818,0.250000,0.177778,0.207407
3,4,2018-04-26,2018-05-10,0.545455,0.694444,0.642857,0.570707
4,5,2018-05-11,2018-05-25,0.000000,0.000000,0.000000,0.000000
5,mean,NaN,NaN,0.236364,0.314815,0.325238,0.256691
6,median,NaN,NaN,0.181818,0.250000,0.388889,0.216450


,fold,test_start,test_end,accuracy,precision_macro,recall_macro,f1_macro
0,1,2018-03-09,2018-03-23,0.272727,0.090909,0.333333,0.142857
1,2,2018-03-26,2018-04-10,0.090909,0.030303,0.333333,0.055556
2,3,2018-04-11,2018-04-25,0.181818,0.095238,0.133333,0.111111
3,4,2018-04-26,2018-05-10,0.181818,0.066667,0.333333,0.111111
4,5,2018-05-11,2018-05-25,0.545455,0.181818,0.333333,0.235294
5,mean,NaN,NaN,0.254545,0.092987,0.293333,0.131186
6,median,NaN,NaN,0.181818,0.090909,0.333333,0.111111


,fold,test_start,test_end,accuracy,precision_macro,recall_macro,f1_macro
0,1,2018-03-09,2018-03-23,0.181818,0.060606,0.333333,0.102564
1,2,2018-03-26,2018-04-10,0.363636,0.236111,0.305556,0.240741
2,3,2018-04-11,2018-04-25,0.272727,0.222222,0.333333,0.264550
3,4,2018-04-26,2018-05-10,0.545455,0.428571,0.523810,0.390572
4,5,2018-05-11,2018-05-25,0.272727,0.090909,0.333333,0.142857
5,mean,NaN,NaN,0.327273,0.207684,0.365873,0.228257
6,median,NaN,NaN,0.272727,0.222222,0.333333,0.240741


,fold,test_start,test_end,accuracy,precision_macro,recall_macro,f1_macro
0,1,2018-03-09,2018-03-23,0.181818,0.060606,0.333333,0.102564
1,2,2018-03-26,2018-04-10,0.545455,0.181818,0.333333,0.235294
2,3,2018-04-11,2018-04-25,0.272727,0.188889,0.244444,0.207407
3,4,2018-04-26,2018-05-10,0.454545,0.416667,0.476190,0.333333
4,5,2018-05-11,2018-05-25,0.181818,0.066667,0.333333,0.111111
5,mean,NaN,NaN,0.327273,0.182929,0.344127,0.197942
6,median,NaN,NaN,0.272727,0.181818,0.333333,0.207407


## Notes
- These runs avoid GridSearchCV to keep the demo fast. For your final report, you can cite GridSearchCV results from the app logs.
- Use the same date range and horizon as your Streamlit app to align results.
- You can extend this notebook to plot predictions vs actuals or confusion matrices as needed.